# Servidor de Qwen2.5 7B en Google Colab para la aplicación web

Este notebook levanta en Colab el mismo servidor que se usó en Khipu: el grafo multiagente con Qwen2.5 7B en 4 bits, el índice FAISS y la búsqueda web. Al final imprime un enlace a la aplicación de GitHub Pages que ya apunta a este servidor, así que se puede usar desde cualquier computadora o celular.

Pasos para usarlo:

1. Entorno de ejecución, Cambiar tipo de entorno de ejecución, GPU T4.
2. Entorno de ejecución, Ejecutar todo.
3. Abrir el enlace que imprime la última celda y presionar Cargar modelo.

Tiempos aproximados en una T4:

| Paso | Primera vez | Con caché en Drive |
|---|---|---|
| Clonar el repositorio e instalar dependencias | 1 a 2 min | 1 a 2 min |
| Índice FAISS (descarga de datasets y embeddings de 10 758 documentos) | 2 a 3 min | segundos |
| Modelo (descarga de 15 GB, cuantización y copia de 5.5 GB a Drive) | 6 a 10 min | 1 a 3 min |
| Servidor y túnel público | 1 min | 1 min |
| Total | 10 a 15 min | 4 a 6 min |

Una consulta con el multiagente tarda entre 60 y 120 s en la T4 (en la A6000 de Khipu tardaba unos 45 s) y una con el Naive RAG unos 10 s. Son estimaciones: dependen de cuántos pasos use el agente.

Sobre guardar el modelo en Drive: copiar los 15 GB originales no conviene, porque leerlos desde Drive suele ser tan lento como descargarlos de Hugging Face. Lo que sí conviene es guardar la versión ya cuantizada en 4 bits (unos 5.5 GB): se lee más rápido y evita cuantizar cada vez. Por eso la opción USAR_DRIVE guarda el modelo cuantizado y el índice, y en las sesiones siguientes los reutiliza.

Límites de Colab: la sesión gratuita se corta tras unos 90 minutos sin interacción o después de varias horas, y la URL del túnel cambia cada vez que se vuelve a ejecutar. Mientras el notebook esté abierto y la celda del túnel siga activa, el servidor responde.

## 1. Configuración y GPU

In [ ]:
USAR_DRIVE = True   # guarda el modelo cuantizado y el índice en Drive para que la próxima sesión arranque más rápido
CARPETA_DRIVE = "/content/drive/MyDrive/auditor_react"
REPO = "https://github.com/ricardoamiel/auditor-react.git"
APP = "https://ricardoamiel.github.io/auditor-react/"

import subprocess, time, os, json, urllib.request, urllib.parse
inicio = time.time()
def paso(texto):
    print(f"[{(time.time() - inicio) / 60:4.1f} min] {texto}")

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout
      or "Sin GPU: cambia el entorno de ejecución a GPU T4")

## 2. Código y dependencias

In [ ]:
if not os.path.exists("/content/proyecto"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/proyecto"], check=True)
else:
    subprocess.run(["git", "-C", "/content/proyecto", "pull", "-q"], check=True)
%cd /content/proyecto
%pip install -q langgraph faiss-cpu sentence-transformers bitsandbytes ddgs fastapi uvicorn
paso("código y dependencias listos")

## 3. Índice FAISS

Si ya está en Drive se copia; si no, se construye con codes/prepare_data.py y se guarda para la próxima vez.

In [ ]:
import shutil
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CARPETA_DRIVE, exist_ok=True)

indice_drive = f"{CARPETA_DRIVE}/index"
if USAR_DRIVE and os.path.exists(f"{indice_drive}/vectores.npy"):
    shutil.copytree(indice_drive, "data/index", dirs_exist_ok=True)
    paso("índice copiado desde Drive")
else:
    subprocess.run(["python", "codes/prepare_data.py", "--solo-indice"], check=True)
    if USAR_DRIVE:
        shutil.copytree("data/index", indice_drive, dirs_exist_ok=True)
    paso("índice construido")

## 4. Modelo

Con USAR_DRIVE, la primera vez se descarga Qwen2.5 7B, se cuantiza en 4 bits en un proceso aparte y se guarda en Drive; las siguientes veces se carga directo desde ahí. Sin Drive, el servidor lo descarga de Hugging Face al arrancar.

In [ ]:
modelo_drive = f"{CARPETA_DRIVE}/qwen2.5-7b-nf4"
if USAR_DRIVE:
    if not os.path.exists(f"{modelo_drive}/config.json"):
        guardar = (
            "import sys; sys.path.insert(0, 'codes')\n"
            "from llm import LLM\n"
            "m = LLM('Qwen/Qwen2.5-7B-Instruct')\n"
            f"m.model.save_pretrained('{modelo_drive}'); m.tok.save_pretrained('{modelo_drive}')\n"
        )
        subprocess.run(["python", "-c", guardar], check=True)  # proceso aparte: libera la GPU al terminar
        paso("modelo cuantizado y guardado en Drive")
    MODELO = modelo_drive
else:
    MODELO = "Qwen/Qwen2.5-7B-Instruct"
print("Modelo:", MODELO)

## 5. Servidor

Se lanza app/server.py en segundo plano y se espera a que responda. El registro queda en servidor.log.

In [ ]:
entorno = {**os.environ, "MODELO_LLM": MODELO, "TOKENIZERS_PARALLELISM": "false"}
servidor = subprocess.Popen(["python", "app/server.py"], stdout=open("servidor.log", "w"), stderr=subprocess.STDOUT, env=entorno)
while True:
    try:
        info = json.loads(urllib.request.urlopen("http://localhost:8000/salud", timeout=5).read())
        break
    except Exception:
        if servidor.poll() is not None:
            print(open("servidor.log").read()[-3000:])
            raise RuntimeError("El servidor se detuvo; el registro está arriba")
        time.sleep(10)
paso(f"servidor listo con {info['modelo']} en {info['gpu']}")

## 6. Túnel público y enlace a la aplicación

GitHub Pages se sirve por https, así que el servidor necesita una URL https pública. Cloudflare la da sin cuenta (trycloudflare). Esta celda imprime el enlace que abre la aplicación ya configurada.

In [ ]:
import re
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "-O", "cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("cloudflared", 0o755)
tunel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=open("tunel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunel.log").read())
    if m:
        URL = m.group(0)
        break
    time.sleep(2)
assert URL, "No se obtuvo la URL del túnel; revisa tunel.log"
paso("túnel activo")
print("URL del servidor:", URL)
print("Abre la aplicación ya configurada:", APP + "?servidor=" + urllib.parse.quote(URL, safe=""))

## 7. Prueba rápida

Una consulta en modo Naive RAG a través del túnel, para confirmar que todo el camino funciona.

In [ ]:
time.sleep(5)  # el DNS del túnel tarda unos segundos en propagarse
pedido = urllib.request.Request(URL + "/chat", data=json.dumps({"pregunta": "In which country is Machu Picchu located?", "modo": "naive"}).encode(),
                                headers={"Content-Type": "application/json"})
t0 = time.time()
lineas = urllib.request.urlopen(pedido, timeout=300).read().decode().strip().splitlines()
print(json.loads(lineas[-1])["respuesta"])
print(f"{time.time() - t0:.1f} s")

## 8. Mantener el servidor activo

Esta celda mantiene viva la sesión y muestra el uso de memoria cada cinco minutos. Para apagar todo, detener la celda y luego Entorno de ejecución, Desconectar y borrar el entorno.

In [ ]:
try:
    while servidor.poll() is None:
        uso = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
        paso(f"servidor activo en {URL} | GPU {uso}")
        time.sleep(300)
    print(open("servidor.log").read()[-2000:])
except KeyboardInterrupt:
    servidor.terminate(); tunel.terminate()
    print("Servidor y túnel detenidos")